In [35]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

In [36]:
texts = [
    "I love this movie",
    "This film is amazing",
    "Very good acting",
    "Excellent story",
    "I hate this movie",
    "Terrible film",
    "Very boring story",
    "Worst acting ever"
]

In [37]:
labels=np.array([1,1,1,1,
                 0,0,0,0])

In [38]:
vocab_size=1000
max_length=6
tokenizer=Tokenizer(num_words=vocab_size,oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequences=tokenizer.texts_to_sequences(texts)
x=pad_sequences(sequences,maxlen=max_length,padding="post")
print("Word Index")
print(tokenizer.word_index)
print("\nInput Sequence")
print(x)

Word Index
{'<OOV>': 1, 'this': 2, 'i': 3, 'movie': 4, 'film': 5, 'very': 6, 'acting': 7, 'story': 8, 'love': 9, 'is': 10, 'amazing': 11, 'good': 12, 'excellent': 13, 'hate': 14, 'terrible': 15, 'boring': 16, 'worst': 17, 'ever': 18}

Input Sequence
[[ 3  9  2  4  0  0]
 [ 2  5 10 11  0  0]
 [ 6 12  7  0  0  0]
 [13  8  0  0  0  0]
 [ 3 14  2  4  0  0]
 [15  5  0  0  0  0]
 [ 6 16  8  0  0  0]
 [17  7 18  0  0  0]]


In [39]:
class TokenAndPositionEmbedding(layers.Layer): #1
    def __init__(self, max_length, vocab_size, embed_dim):
        super().__init__()
        self.token_embedding = layers.Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim
        )
 
        self.position_embedding = layers.Embedding(
            input_dim=max_length,
            output_dim=embed_dim
        )
 
    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)
        token_emb = self.token_embedding(x)
        position_emb = self.position_embedding(positions)
        return token_emb + position_emb
 
 

In [40]:
class TransformerBlock(layers.Layer):

    def __init__(self, embed_dim, num_heads, ff_dim):

        super().__init__()

        self.attention = layers.MultiHeadAttention(

            num_heads=num_heads,

            key_dim=embed_dim

        )

 

        self.ffn = tf.keras.Sequential([

            layers.Dense(ff_dim, activation="relu"),

            layers.Dense(embed_dim)

        ])

 

        self.layernorm1 = layers.LayerNormalization()

        self.layernorm2 = layers.LayerNormalization()

 

    def call(self, inputs):

        # Self-attention (Query, Key, Value are calculated here using the same input)

        # The attention layer takes the input and computes

        # attention scores to capture relationships between different positions in the sequence.

        attention_output = self.attention(inputs, inputs)

 

        # Add + Normalize

        out1 = self.layernorm1(inputs + attention_output)

 

        # Feed-forward network

        ffn_output = self.ffn(out1)

 

        # Add + Normalize

        out2 = self.layernorm2(out1 + ffn_output)

 

        return out2




In [41]:
# build the model

embed_dim = 16

num_heads = 2

ff_dim = 32

inputs = layers.Input(shape=(max_length,))

 

x = TokenAndPositionEmbedding(

    max_length=max_length,

    vocab_size=vocab_size,

    embed_dim=embed_dim

)(inputs)

 

x = TransformerBlock(

    embed_dim=embed_dim,

    num_heads=num_heads,

    ff_dim=ff_dim

)(x)

 

x = layers.GlobalAveragePooling1D()(x)

 

outputs = layers.Dense(1, activation="sigmoid")(x)

 

model = tf.keras.Model(inputs=inputs, outputs=outputs)




In [42]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]

)
model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 6)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_3  │ (None, 6, 16)          │        16,096 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_2             │ (None, 6, 16)          │         3,296 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_2      │ (None, 16)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,409 (75.82 KB)

 Trainable params: 19,409 (75.82 KB)

 Non-trainable params: 0 (0.00 B)

In [43]:

print(X.shape)
print(labels.shape)

(8, 6)
(8,)


In [44]:
model.fit(
    X,
    labels,
    epochs=30,
    batch_size=2,
    verbose=1
)

Epoch 1/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 14s 87ms/step - accuracy: 0.6250 - loss: 0.7053
Epoch 2/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5000 - loss: 0.7062 
Epoch 3/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.5000 - loss: 0.6463
Epoch 4/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.7500 - loss: 0.5720
Epoch 5/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.7500 - loss: 0.5540
Epoch 6/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8750 - loss: 0.5074
Epoch 7/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8750 - loss: 0.4961
Epoch 8/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 1.0000 - loss: 0.4665 
Epoch 9/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 1.0000 - loss: 0.4458
Epoch 10/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 1.0000 - loss: 0.4279
Epoch 11/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.4099
Epoch 12/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 1.0000 - loss: 0.387

In [ ]:
test_sentences = [
    "I love the film",
    "This movie was awful"
]
 
test_seq = tokenizer.texts_to_sequences(test_sentences)
test_pad = pad_sequences(test_seq, maxlen=max_length, padding="post")
predictions = model.predict(test_pad)
 
for sentence, prediction in zip(test_sentences, predictions):
    print(sentence, "->", prediction[0])
    if prediction[0] > 0.5:
        print("Prediction: Positive")
    else:
        print("Prediction: Negative")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 483ms/step
I love the film -> 0.82448494
Prediction: Positive
This movie was awful -> 0.83055043
Prediction: Positive
